# Demo: rendering a full match to video

This notebook plays one full match between two trained policies using the real `Vasuki` environment and its `render()` method, then writes the rendered frames to an `.avi` video file in this `demo/` directory using OpenCV's `cv2.VideoWriter`.

**Requirements / assumptions:**

- This notebook requires trained model artifacts to exist on disk. By default it looks for:
  - `models/dqn_random.zip` (produced by `python -m training.train_dqn`)
  - `models/dqn_selfplay.zip` (produced by `python -m training.train_selfplay`)
- If one or both files are missing, the notebook falls back to `RandomOpponent` for that side so the demo can still run end-to-end (with a printed warning) before you have trained models.
- Output video is written to `demo/match_demo.avi`.
- `Vasuki.render(actionA, actionB)` returns a single BGR frame as a `numpy.ndarray` of shape `(scale*n, 2*scale*n, 3)` (uint8-range float values in `[0, 255]`), built from the **previous** step's state (it reads `self.history[-2]`). This means `render()` can only be called from the second `step_two` call onward (i.e. once `len(env.history) >= 2`).

In [ ]:
from pathlib import Path

import numpy as np
import cv2
from stable_baselines3 import DQN

from vasuki.env import Vasuki
from vasuki.opponents import RandomOpponent, PolicySnapshotOpponent

CONFIG = {"n": 8, "rewards": {"Food": 4, "Movement": -1, "Illegal": -2}, "game_length": 100}

MODELS_DIR = Path("models")
DEMO_DIR = Path("demo")
DQN_RANDOM_PATH = MODELS_DIR / "dqn_random.zip"
DQN_SELFPLAY_PATH = MODELS_DIR / "dqn_selfplay.zip"
OUTPUT_VIDEO_PATH = DEMO_DIR / "match_demo.avi"
FPS = 10

In [ ]:
def _load_agent_or_fallback(path, label):
    if path.exists():
        return PolicySnapshotOpponent(DQN.load(str(path)))
    print(f"Warning: {path} not found, falling back to RandomOpponent for {label}")
    return RandomOpponent()


agent_a = _load_agent_or_fallback(DQN_RANDOM_PATH, "agent A (expected dqn_random)")
agent_b = _load_agent_or_fallback(DQN_SELFPLAY_PATH, "agent B (expected dqn_selfplay)")

## Runner helper

A small helper class that drives a `Vasuki` match to completion, calling `render()` after every step (once history has at least 2 entries) and collecting frames for video export. `opp_b` is applied to agent B by temporarily swapping `env.agentA`/`env.agentB`, matching the convention used in `analysis/evaluate.py`'s `_act_as_b` and `vasuki/gym_wrapper.py`'s `_opponent_action`.

In [ ]:
class Runner:
    """Drives one full Vasuki match and collects rendered frames."""

    def __init__(self, config, opp_a, opp_b):
        self.env = Vasuki(**config)
        self.opp_a = opp_a
        self.opp_b = opp_b
        self.frames = []

    def _act_as_b(self):
        env = self.env
        env.agentA, env.agentB = env.agentB, env.agentA
        try:
            return self.opp_b.act(env)
        finally:
            env.agentA, env.agentB = env.agentB, env.agentA

    def run(self):
        self.env.reset()
        done = False
        while not done:
            action_a = self.opp_a.act(self.env)
            action_b = self._act_as_b()
            _, _, done, _ = self.env.step_two(action_a, action_b)
            if len(self.env.history) >= 2:
                frame = self.env.render(action_a, action_b)
                self.frames.append(frame.astype(np.uint8))
        return self.env


runner = Runner(CONFIG, agent_a, agent_b)
final_env = runner.run()
print(f"Match finished. Collected {len(runner.frames)} frames.")
print(f"Final score - A: {final_env.agentA['score']}, B: {final_env.agentB['score']}")

## Write frames to video

Uses `cv2.VideoWriter` with the `XVID` codec to write an `.avi` file into `demo/`.

In [ ]:
DEMO_DIR.mkdir(parents=True, exist_ok=True)

if not runner.frames:
    print("No frames were collected (match may have been too short); skipping video export.")
else:
    height, width = runner.frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    writer = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, FPS, (width, height))
    try:
        for frame in runner.frames:
            writer.write(frame)
    finally:
        writer.release()
    print(f"Wrote {len(runner.frames)} frames to {OUTPUT_VIDEO_PATH}")